In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "userdata.get('OPENAI_API_KEY')"

In [ ]:
# =============================================
# 0) تثبيت المكتبات (مرة واحدة في كولاب)
# =============================================
# شغّل هذا السطر لو ما كانت المكتبات منصّبة
# !pip install -q faiss-cpu sentence-transformers transformers accelerate

# =============================================
# 1) المسارات العامة و الإعدادات
# =============================================
!pip install faiss-cpu sentence-transformers transformers accelerate openai

import os
import json
import numpy as np
import faiss
import tensorflow as tf
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from openai import OpenAI

from tensorflow.keras.preprocessing import image as kimage

# ------------ عدّل هذه المسارات حسب مشروعك ------------
BASE_DIR = "/content/drive/MyDrive/data"  # مجلد الصور (كل كلاس = فولدر)
VISION_MODEL_PATH = "/content/drive/MyDrive/skin_models/EfficientNetB3(105).keras"

DISEASES_FILE = "/content/drive/MyDrive/diseases.jsonl"  # ملف الأمراض نصياً
FAISS_INDEX_PATH = "/content/drive/MyDrive/skin_models/disease_faiss.index"
EMB_PATH = "/content/drive/MyDrive/skin_models/disease_embeddings.npy"
META_PATH = "/content/drive/MyDrive/skin_models/disease_meta.json"

# ملف أسماء الكلاسات تبع المودل (اختياري لكن مهم عشان ترتيب الليبل)
CLASSES_PATH = "/content/drive/MyDrive/skin_models/classes.json"

IMG_SIZE = (380, 380)  # نفس حجم الصور في EfficientNetB3
TOP_K = 3              # عدد الوثائق اللي ترجعها من RAG

# اسم موديل FLORA من HuggingFace (عدّله للاسم الحقيقي)
LLM_MODEL_NAME = "gpt-4.1-mini"  # ✅ OpenAI model (تقدر تغيره)


# =============================================
# 2) تحميل بيانات الأمراض من ملف JSONL
#    (id, name, symptoms, causes, treatments, sources, language)
# =============================================
def load_disease_corpus(diseases_path):
    docs = []
    with open(diseases_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            # نص موحد للـ Embedding
            text = (
                f"المرض: {rec.get('name','')}. "
                f"الأعراض: {', '.join(rec.get('symptoms', []))}. "
                f"الأسباب: {', '.join(rec.get('causes', []))}. "
                f"العلاج: {', '.join(rec.get('treatments', []))}. "
                f"المصادر: {', '.join(rec.get('sources', []))}."
            )
            docs.append({
                "id": rec.get("id"),
                "name": rec.get("name"),
                "text": text,
                "raw": rec
            })
    print(f"📄 Loaded {len(docs)} disease records from JSONL.")
    return docs


# =============================================
# 3) إعداد موديل الـ Embedding + بناء / تحميل FAISS Index
# =============================================
# نستخدم موديل متعدد اللغات (يدعم العربي):
EMB_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("🔁 Loading sentence-transformer embedding model on CPU...")
emb_model = SentenceTransformer(EMB_MODEL_NAME, device="cpu")  # 👈 مهم جداً

def build_faiss_index(docs, emb_path=EMB_PATH, index_path=FAISS_INDEX_PATH, meta_path=META_PATH):
    texts = [d["text"] for d in docs]
    print("🧠 Encoding disease texts into embeddings...")
    embeddings = emb_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    # نطبع الأبعاد للتأكد
    print("Embeddings shape:", embeddings.shape)

    # نطبع L2 norm للتطبيع عشان نستخدم cosine similarity
    faiss.normalize_L2(embeddings)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # inner product = cosine بعد التطبيع
    index.add(embeddings)

    # حفظ
    os.makedirs(os.path.dirname(emb_path), exist_ok=True)
    np.save(emb_path, embeddings)
    faiss.write_index(index, index_path)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(docs, f, ensure_ascii=False, indent=2)

    print("✅ FAISS index + embeddings + metadata saved.")
    return index, embeddings, docs


def load_faiss_index(emb_path=EMB_PATH, index_path=FAISS_INDEX_PATH, meta_path=META_PATH):
    if not (os.path.exists(emb_path) and os.path.exists(index_path) and os.path.exists(meta_path)):
        raise FileNotFoundError("❌ FAISS أو الـ embeddings أو metadata غير موجودة. شغّل build_faiss_index أولاً.")

    embeddings = np.load(emb_path)
    index = faiss.read_index(index_path)
    with open(meta_path, "r", encoding="utf-8") as f:
        docs = json.load(f)

    print("✅ Loaded FAISS index & embeddings & metadata.")
    return index, embeddings, docs


# نحاول تحميل FAISS لو موجود، غير هيك نبنيه من الصفر:
disease_docs = load_disease_corpus(DISEASES_FILE)
try:
    faiss_index, disease_embs, disease_docs = load_faiss_index()
except Exception as e:
    print("ℹ️ إعادة بناء FAISS index لأنه غير موجود أو فيه مشكلة:", e)
    faiss_index, disease_embs, disease_docs = build_faiss_index(disease_docs)


# وظيفة retrieve من FAISS:
def retrieve_diseases(query_text, top_k=TOP_K):
    q_emb = emb_model.encode([query_text], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = faiss_index.search(q_emb, top_k)
    idxs = I[0]
    scores = D[0]
    results = []
    for i, s in zip(idxs, scores):
        doc = disease_docs[int(i)]
        results.append((doc, float(s)))
    return results


# =============================================
# 4) تحميل موديل الصور EfficientNetB3
# =============================================
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import Precision, Recall, AUC

# لو كنت مستخدم focal_loss مخصص لازم تعرّفه هنا
def focal_loss_fixed(y_true, y_pred, gamma=1.5, alpha=0.45):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    cross_entropy = -y_true * tf.math.log(y_pred)
    weight = alpha * tf.math.pow(1 - y_pred, gamma)
    loss = weight * cross_entropy
    return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

custom_objects = {
    "focal_loss_fixed": focal_loss_fixed,
    "Precision": Precision,
    "Recall": Recall,
    "AUC": AUC,
}

print("🔁 Loading vision model (EfficientNetB3)...")
vision_model = load_model(VISION_MODEL_PATH, custom_objects=custom_objects)
vision_model.trainable = False
print("✅ Vision model loaded.")

# تحميل أسماء الكلاسات
if os.path.exists(CLASSES_PATH):
    with open(CLASSES_PATH, "r", encoding="utf-8") as f:
        class_names = json.load(f)
else:
    # fallback لو ما عندك ملف
    class_names = ["Acne", "Eczema", "Psoriasis","Warts","vitiligo"]
    print("⚠️ CLASSES_PATH غير موجود. استخدمت ترتيب افتراضي:", class_names)


def preprocess_image_for_vision(img_path):
    img = kimage.load_img(img_path, target_size=IMG_SIZE)
    arr = kimage.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = tf.keras.applications.efficientnet.preprocess_input(arr)
    return arr, img  # arr للمدخل، img للعرض


def predict_disease_from_image(img_path):
    x, pil_img = preprocess_image_for_vision(img_path)
    preds = vision_model.predict(x)
    idx = int(np.argmax(preds[0]))
    conf = float(np.max(preds[0]))
    disease_id = class_names[idx]
    return disease_id, conf, pil_img


# =============================================
# 5) تحميل موديل FLORA (LLM) من HuggingFace
#     ✅ التغيير الوحيد: بدل HuggingFace نستعمل OpenAI
#     ✅ ونحافظ على نفس الواجهة: gen_pipe(...) ترجع [{"generated_text": "..."}]
# =============================================
print("🔁 Loading gpt-4.1-mini...")

LLM_MODEL_NAME = "gpt-4.1-mini"  # ← OpenAI model

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def gen_pipe(
    prompt,
    max_new_tokens=None,
    do_sample=False,
    temperature=0.0,
    top_p=1.0,
    num_return_sequences=1,
    **kwargs
):
    # OpenAI Chat Completion
    resp = client.chat.completions.create(
        model=LLM_MODEL_NAME,
        messages=[
            {"role": "system", "content": "أنت مساعد افتراضي متخصص في الأمراض الجلدية."},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_new_tokens,
        top_p=top_p,
    )
    text = resp.choices[0].message.content
    return [{"generated_text": text}]

print("✅gpt-4.1-mini  جاهز للاستخدام.")


# =============================================
# 6) دالة توليد الإجابة من RAG + LLM
# =============================================
def build_context_block(retrieved_docs):
    """
    يحوّل النتائج من FAISS إلى نص context منظم نحطه في الـ prompt.
    """
    lines = []
    for i, (doc, score) in enumerate(retrieved_docs, start=1):
        raw = doc["raw"]
        lines.append(
            f"📌 ({i}) المرض: {raw.get('name','')}\n"
            f"    الأعراض: {', '.join(raw.get('symptoms', []))}\n"
            f"    الأسباب: {', '.join(raw.get('causes', []))}\n"
            f"    العلاج: {', '.join(raw.get('treatments', []))}\n"
            f"    المصادر: {', '.join(raw.get('sources', []))}\n"
        )
    return "\n".join(lines)


def rag_llm_answer(
    symptom_text: str = "",
    predicted_disease_id: str = None,
    top_k: int = TOP_K
):
    """
    - يستخدم الـ Embeddings + FAISS لاسترجاع أقرب أمراض.
    - يستخدم FLORA لكتابة جواب طبي لطيف بالعربي.
    """

    # تجهيز query للنص
    query_parts = []
    if predicted_disease_id is not None:
        query_parts.append(f"المرض المتوقّع: {predicted_disease_id}")
    if symptom_text:
        query_parts.append(f"الأعراض: {symptom_text}")

    if not query_parts:
        query_text = "استعلام عام عن أمراض الجلد"
    else:
        query_text = " | ".join(query_parts)

    # استرجاع من FAISS
    retrieved = retrieve_diseases(query_text, top_k=top_k)
    context_block = build_context_block(retrieved)

    # اسم المرض المتوقع بلغة جميلة
    pred_name_human = predicted_disease_id if predicted_disease_id else "غير محدد"

    # نبني الـ prompt بالعربي
    prompt = f"""
أنت مساعد افتراضي متخصص في الأمراض الجلدية.
لديك قاعدة معرفة صغيرة، وهذه مقتطفات منها:

{context_block}

معلومات من نموذج الرؤية (تصنيف الصور):
- المرض المتوقّع: {pred_name_human}

معلومات من المريض (إن وُجدت):
- الأعراض المبلغ عنها: {symptom_text if symptom_text else "لم يذكر المريض أعراضاً نصية، الاعتماد على الصورة فقط."}

المطلوب منك:
- اقتراح أقرب مرض (أو 2–3 أمراض محتملة إن لزم).
- شرح بسيط عن المرض.
- ذكر أهم الأعراض بالنقاط.
- ذكر أبرز الأسباب المحتملة.
- ذكر أهم العلاجات الشائعة (بدون وصف جرعات أو أدوية محددة بالاسم التجاري إن أمكن).
- تذكير المريض بضرورة مراجعة طبيب جلدية للفحص المباشر وعدم الاعتماد على النظام وحده.

جاوب بالعربية الفصحى، وبشكل مختصر ومنظم، مستخدماً عناوين فرعية ونقاط.
لا تذكر أنك نموذج لغوي، فقط قدّم النص الطبي التعليمي.
"""
def rag_llm_answer(
    symptom_text: str = "",
    predicted_disease_id: str = None,
    top_k: int = TOP_K
):
    query_parts = []
    if predicted_disease_id is not None:
        query_parts.append(f"المرض المتوقّع: {predicted_disease_id}")
    if symptom_text:
        query_parts.append(f"الأعراض: {symptom_text}")

    query_text = " | ".join(query_parts) if query_parts else "استعلام عام عن أمراض الجلد"

    retrieved = retrieve_diseases(query_text, top_k=top_k)
    context_block = build_context_block(retrieved)

    prompt = f"""
أنت مساعد افتراضي متخصص في الأمراض الجلدية.

{context_block}

المرض المتوقع: {predicted_disease_id}
الأعراض: {symptom_text}
"""

    out = gen_pipe(prompt)[0]["generated_text"]

    if out is None:
        out = ""

    if prompt in out:
        out = out.split(prompt, 1)[-1].strip()

    return out   # ✅ هذا هو السطر اللي كان ناقص
# =============================================
# 7) دالة كاملة: صورة + (اختياري نص أعراض) → تشخيص + تقرير
# =============================================
def analyze_image_with_rag_llm(img_path, symptom_text: str = ""):
    """
    1) يتنبأ بالمرض من الصورة باستخدام EfficientNetB3.
    2) يستخدم RAG + FLORA لإنتاج تقرير عن المرض.
    """
    disease_id, conf, pil_img = predict_disease_from_image(img_path)
    print(f"🩺 Vision Model Prediction: {disease_id} (confidence = {conf*100:.2f}%)")

    report = rag_llm_answer(symptom_text=symptom_text, predicted_disease_id=disease_id)

    print("\n============================")
    print("📄 تقرير LLM + RAG:")
    print("============================\n")
    print(report)
    return disease_id, conf, report


# =============================================
# 😎 مثال استخدام
# =============================================

# مثال: صورة من vitiligo
# غيّر هذا المسار لصورة حقيقية من بياناتك:
# example_image = "/content/drive/MyDrive/data/vitiligo/xxx.jpg"
# _ = analyze_image_with_rag_llm(example_image, symptom_text="بقع بيضاء غير مؤلمة في اليدين والوجه.")

🔁 Loading sentence-transformer embedding model on CPU...
📄 Loaded 5 disease records from JSONL.
✅ Loaded FAISS index & embeddings & metadata.
🔁 Loading vision model (EfficientNetB3)...
✅ Vision model loaded.
⚠️ CLASSES_PATH غير موجود. استخدمت ترتيب افتراضي: ['Acne', 'Eczema', 'Psoriasis', 'Warts', 'vitiligo']
🔁 Loading gpt-4.1-mini...
✅gpt-4.1-mini  جاهز للاستخدام.


In [ ]:


test_image_path = "/content/drive/MyDrive/splitted_data/test/vitiligo/istockphoto-1221650885-612x612.jpg"  # عدّل المسار حسب داتا الاختبار عندك

disease_id, conf, report = analyze_image_with_rag_llm(
    test_image_path,
)


print("\n🔍 Disease ID:", disease_id)
print("🔢 Confidence:", f"{conf*100:.2f}%")
print("\n🩺 Final Report:\n", report)

1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
🩺 Vision Model Prediction: vitiligo (confidence = 99.56%)

📄 تقرير LLM + RAG:

📌 (1) المرض: البهاق (Vitiligo)  
الأعراض:  
- فقدان لون الجلد في بقع بيضاء غير منتظمة على الجلد.  
- قد تظهر البقع على أي جزء من الجسم، وغالبًا ما تبدأ على اليدين، الوجه، أو حول الفم والعينين.  
- فقدان لون الشعر على فروة الرأس، الحواجب، أو الرموش.  
- فقدان لون في الأنسجة الداخلية للفم والأنف.  

الأسباب:  
- اضطراب مناعي ذاتي حيث يهاجم الجهاز المناعي الخلايا الصبغية (الخلايا التي تنتج الميلانين).  
- عوامل وراثية قد تلعب دورًا.  
- التعرض لبعض المواد الكيميائية.  
- قد يرتبط ببعض الأمراض الأخرى مثل أمراض الغدة الدرقية.  

العلاج:  
- لا يوجد علاج شافٍ، لكن هناك خيارات لتحسين مظهر الجلد مثل:  
  - استخدام كريمات الستيرويد الموضعية.  
  - العلاج بالضوء (Phototherapy).  
  - في بعض الحالات، زراعة الجلد أو إزالة الصبغة من الجلد السليم لتوحيد اللون.  
  - استخدام مستحضرات التجميل لتغطية البقع.  

المصادر:  
https://www.mayoclinic.org/diseases-conditions/vitiligo

🔍 Disease 